# ARC Prize 2026 — Evaluación (kernel GPU, offline)

Mide **pass@2** en `evaluation` con el pipeline de producción: **1 worker por GPU**
(sharding, `arc.runner`), TTT por tarea y selección de attempts por **scoring de
log-verosimilitud augmentada** (`selection='score'`). Adjuntar: código (`arc-agi2-code`),
competencia, modelo base y el adaptador (`arc-agi2-model`).

Escribe `eval_score.json` + `submission_eval.json` en `/kaggle/working`.

In [ ]:
import os, sys, glob, json
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# --- Config del experimento (registrar en docs/RESULTS.md al terminar) ---
EVAL_LIMIT = 24        # nº de tareas de evaluation (0 = las 120)
SELECTION = 'score'    # 'score' (nuevo) vs 'vote' (legado) — para el ablation
N_AUG = 4              # vistas augmentadas de generación
N_CANDS = 2            # 1 greedy + 1 sampleado por vista
SCORE_VIEWS = 4
USE_TTT = True
TTT_STEPS = 16
MAX_NEW_TOKENS = 1024
TIME_BUDGET_S = int(1.5 * 3600)  # presupuesto por worker

print('INPUTS:', glob.glob('/kaggle/input/*'))
arc_init = glob.glob('/kaggle/input/**/src/arc/__init__.py', recursive=True)
assert arc_init, 'falta dataset de codigo'
SRC_DIR = os.path.dirname(os.path.dirname(arc_init[0])); sys.path.insert(0, SRC_DIR)
SCRIPTS_DIR = os.path.join(os.path.dirname(SRC_DIR), 'scripts')
WORKER = os.path.join(SCRIPTS_DIR, 'infer_worker.py')
assert os.path.isfile(WORKER), f'falta {WORKER} (re-sube el dataset de codigo)'
ch = glob.glob('/kaggle/input/**/arc-agi_evaluation_challenges.json', recursive=True)
assert ch, 'faltan datos de la competencia'
EVAL_CHALLENGES = ch[0]
COMP_DIR = os.path.dirname(EVAL_CHALLENGES)

# base model (config.json + pesos, sin adapter) y adaptador LoRA (adapter_config.json)
MODEL_DIR = None
for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(cfg); fs = os.listdir(d)
    if any(f.endswith(('.safetensors', '.bin')) for f in fs) and 'adapter_config.json' not in fs:
        MODEL_DIR = d; break
ad = glob.glob('/kaggle/input/**/adapter_config.json', recursive=True)
LORA_DIR = os.path.dirname(ad[0]) if ad else None
assert MODEL_DIR and LORA_DIR, f'MODEL_DIR={MODEL_DIR} LORA_DIR={LORA_DIR}'
print('MODEL_DIR =', MODEL_DIR); print('LORA_DIR =', LORA_DIR); print('COMP_DIR =', COMP_DIR)

In [ ]:
# === Lanzar 1 worker por GPU sobre el split de evaluation ===
import subprocess, time
from pathlib import Path
import torch

N_WORKERS = max(1, torch.cuda.device_count())
shard_paths = [f'/kaggle/working/eval_shard_{i}.json' for i in range(N_WORKERS)]
procs, logs = [], []
for i in range(N_WORKERS):
    cmd = [sys.executable, WORKER,
           '--challenges', EVAL_CHALLENGES, '--out', shard_paths[i],
           '--n-shards', str(N_WORKERS), '--shard-idx', str(i),
           '--model-dir', MODEL_DIR, '--lora-dir', LORA_DIR,
           '--selection', SELECTION, '--n-aug', str(N_AUG),
           '--n-cands', str(N_CANDS), '--score-views', str(SCORE_VIEWS),
           '--max-new-tokens', str(MAX_NEW_TOKENS),
           '--time-budget-s', str(TIME_BUDGET_S)]
    if EVAL_LIMIT:
        cmd += ['--limit', str(EVAL_LIMIT)]
    if USE_TTT:
        cmd += ['--ttt', '--ttt-steps', str(TTT_STEPS)]
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(i), PYTHONUNBUFFERED='1')
    lf = open(f'/kaggle/working/eval_worker_{i}.log', 'w')
    procs.append(subprocess.Popen(cmd, env=env, stdout=lf, stderr=subprocess.STDOUT))
    logs.append(lf)
print(f'{N_WORKERS} workers lanzados')

deadline = time.time() + TIME_BUDGET_S + 900
while any(p.poll() is None for p in procs):
    if time.time() > deadline:
        print('Deadline: terminando workers (shards incrementales).')
        for p in procs:
            if p.poll() is None:
                p.terminate()
        break
    time.sleep(60)
    print(f'[{time.strftime("%H:%M:%S")}] vivos:',
          sum(p.poll() is None for p in procs), '/', N_WORKERS, flush=True)
for p in procs:
    p.wait(timeout=120)
for lf in logs:
    lf.close()
for i in range(N_WORKERS):
    tail = Path(f'/kaggle/working/eval_worker_{i}.log').read_text(encoding='utf-8').splitlines()[-6:]
    print(f'--- worker {i} ---'); print('\n'.join(tail))

In [ ]:
# === Merge + score pass@2 ===
from arc.data import load_split
from arc.metrics import score_submission
from arc.runner import merge_submissions

all_tasks = load_split('evaluation', data_dir=COMP_DIR)
tasks = dict(list(all_tasks.items())[:EVAL_LIMIT]) if EVAL_LIMIT else all_tasks
solutions = {t: [p.output for p in task.test] for t, task in tasks.items()}

submission = merge_submissions(shard_paths, tasks)
submission = {t: submission[t] for t in tasks}
score = score_submission(submission, solutions)
print(f'\n=== SCORE pass@2 en {len(tasks)} tareas | selection={SELECTION} | '
      f'TTT={USE_TTT} : {score:.4f} ===')

json.dump({'score': score, 'n': len(tasks), 'selection': SELECTION, 'ttt': USE_TTT,
           'n_aug': N_AUG, 'n_cands': N_CANDS, 'score_views': SCORE_VIEWS,
           'ttt_steps': TTT_STEPS, 'max_new_tokens': MAX_NEW_TOKENS},
          open('/kaggle/working/eval_score.json', 'w'))
json.dump(submission, open('/kaggle/working/submission_eval.json', 'w'))
print('Escritos eval_score.json y submission_eval.json')